# 라이브러리 및 데이터 불러오기

In [3]:
import pandas as pd
import numpy as np
import pymysql

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [ ]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_friendrequest`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

C:\Users\kkw53\AppData\Local\Temp\ipykernel_15548\844067540.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,status,created_at,updated_at,receive_user_id,send_user_id
0,7,P,2023-04-17 18:29:11,2023-04-17 18:29:11,831962,837521
1,10,A,2023-04-17 18:29:11,2023-04-22 06:02:53,832151,837521
2,11,A,2023-04-17 18:29:11,2023-04-18 19:28:41,832340,837521
3,13,A,2023-04-17 18:29:11,2023-04-19 11:05:04,833041,837521
4,20,P,2023-04-17 18:29:11,2023-04-17 18:29:11,834415,837521


- 신청시 created_at이 생성되고 동일한 값으로 updated_at이 생성되며, 상태(status)값이 업데이트 되는 순간 updated_at이 수정되는 것으로 보임.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17147175 entries, 0 to 17147174
Data columns (total 6 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               int64         
 1   status           str           
 2   created_at       datetime64[us]
 3   updated_at       datetime64[us]
 4   receive_user_id  int64         
 5   send_user_id     int64         
dtypes: datetime64[us](2), int64(3), str(1)
memory usage: 801.3 MB


- 총 17,147,175 행 확인.
- 날짜형태도 결측없이 존재하는 것으로 특이값은 없다고 판단.

## 결측 체크
- 결측 없음.

In [7]:
df.isna().sum()

id                 0
status             0
created_at         0
updated_at         0
receive_user_id    0
send_user_id       0
dtype: int64

## 중복 체크
- 중복 없음.
- 유저아이디 기준 친구 요청 및 요청 수신 유저 모두 요청 건수에 따라서 중복되는 결과 확인. 따로 처리하지 않음.

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df['send_user_id'].nunique()

649072

## 컬럼별 특이값 확인

In [10]:
df.columns

Index(['id', 'status', 'created_at', 'updated_at', 'receive_user_id',
       'send_user_id'],
      dtype='str')

In [11]:
df['status'].value_counts()

status
A    12878407
P     3938608
R      330160
Name: count, dtype: int64

- A (수락), P (대기), R (거절) 범주 이상없음.

In [12]:
print(df['created_at'].min())
print(df['created_at'].max())

2023-04-17 18:29:11
2024-05-09 09:21:47


In [15]:
print(f"보낸 유저 아이디 중 가장 낮은 유저 아이디 : {df['send_user_id'].min()}")
print(f"보낸 유저 아이디 중 가장 큰 유저 아이디 : {df['send_user_id'].max()}")

print(f"수신 유저 아이디 중 가장 낮은 유저 아이디 : {df['receive_user_id'].min()}")
print(f"수신 유저 아이디 중 가장 큰 유저 아이디 : {df['receive_user_id'].max()}")

보낸 유저 아이디 중 가장 낮은 유저 아이디 : 831962
보낸 유저 아이디 중 가장 큰 유저 아이디 : 1583732
수신 유저 아이디 중 가장 낮은 유저 아이디 : 831962
수신 유저 아이디 중 가장 큰 유저 아이디 : 1583731


- 숫자형 아이디의 범위가 모두 유저 테이블에 부합한다고 판단. 특이값 없음.

## 정리
- 모든 테이블의 날짜 데이터의 기준은 확인필요.
- 다른 전처리 작업은 없음. 그대로 유지하는 것을 결론으로 수정 및 처리 코딩없이 유지함.